In [1]:
import pandas as pd

In [12]:
df = pd.read_csv('data/03-28_to_04-27_rm354_FPB.csv')

In [13]:
df.head()

,dateTime,pointDisplayName,point_id,unit,dateTimeUtc,pointName,pointKind,value,deviceName,pointField
0,03/29/2026 00:00:00,Discharge Air Flow,673ce924ffc77731fa3ed999,cfm,03/29/2026 05:00:00,AirFl,Number,1110.425659,FPB-01-018_354,discharge_air_flowrate_sensor
1,03/29/2026 00:00:00,Zone CO2,673ce924ffc77731fa3ed9a5,ppm,03/29/2026 05:00:00,RmCO2,Number,434.000000,FPB-01-018_354,zone_air_co2_concentration_sensor
2,03/29/2026 00:00:30,Zone CO2,673ce924ffc77731fa3ed9a5,ppm,03/29/2026 05:00:30,RmCO2,Number,433.000000,FPB-01-018_354,zone_air_co2_concentration_sensor
3,03/29/2026 00:01:01,Discharge Air Flow,673ce924ffc77731fa3ed999,cfm,03/29/2026 05:01:01,AirFl,Number,1099.938599,FPB-01-018_354,discharge_air_flowrate_sensor
4,03/29/2026 00:01:30,Zone CO2,673ce924ffc77731fa3ed9a5,ppm,03/29/2026 05:01:30,RmCO2,Number,432.000000,FPB-01-018_354,zone_air_co2_concentration_sensor


In [14]:
df['dateTime'] = pd.to_datetime(df['dateTime'])

In [15]:
df = df.sort_values("dateTime").reset_index(drop=True)
df = df.set_index("dateTime")


In [16]:
df.head()

,pointDisplayName,point_id,unit,dateTimeUtc,pointName,pointKind,value,deviceName,pointField
dateTime,,,,,,,,,
2026-03-29 00:00:00,Discharge Air Flow,673ce924ffc77731fa3ed999,cfm,03/29/2026 05:00:00,AirFl,Number,1110.425659,FPB-01-018_354,discharge_air_flowrate_sensor
2026-03-29 00:00:00,Zone CO2,673ce924ffc77731fa3ed9a5,ppm,03/29/2026 05:00:00,RmCO2,Number,434.000000,FPB-01-018_354,zone_air_co2_concentration_sensor
2026-03-29 00:00:30,Zone CO2,673ce924ffc77731fa3ed9a5,ppm,03/29/2026 05:00:30,RmCO2,Number,433.000000,FPB-01-018_354,zone_air_co2_concentration_sensor
2026-03-29 00:01:01,Discharge Air Flow,673ce924ffc77731fa3ed999,cfm,03/29/2026 05:01:01,AirFl,Number,1099.938599,FPB-01-018_354,discharge_air_flowrate_sensor
2026-03-29 00:01:30,Zone CO2,673ce924ffc77731fa3ed9a5,ppm,03/29/2026 05:01:30,RmCO2,Number,432.000000,FPB-01-018_354,zone_air_co2_concentration_sensor


In [17]:
df = df.drop(columns=['pointDisplayName', 'point_id', 'dateTimeUtc', 'pointName', 'pointKind', 'deviceName', 'pointField'])

In [18]:
df.head()

,unit,value
dateTime,,
2026-03-29 00:00:00,cfm,1110.425659
2026-03-29 00:00:00,ppm,434.000000
2026-03-29 00:00:30,ppm,433.000000
2026-03-29 00:01:01,cfm,1099.938599
2026-03-29 00:01:30,ppm,432.000000


In [11]:
start = pd.Timestamp("2026-04-13 07:54:00")
end   = pd.Timestamp("2026-04-13 10:57:00")

df = df.loc[start:end].sort_index()


In [19]:
df.head(50)

,unit,value
dateTime,,
2026-03-29 00:00:00,cfm,1110.425659
2026-03-29 00:00:00,ppm,434.000000
2026-03-29 00:00:30,ppm,433.000000
2026-03-29 00:01:01,cfm,1099.938599
2026-03-29 00:01:30,ppm,432.000000
2026-03-29 00:03:00,ppm,431.000000
2026-03-29 00:04:00,ppm,430.000000
2026-03-29 00:04:30,cfm,1107.480591
2026-03-29 00:05:01,cfm,1097.360352


In [20]:
df = (
    df.pivot_table(index=df.index, columns="unit", values="value", aggfunc="mean")
    .resample("1min").mean()
    .ffill()
    .bfill()
    .rename(columns={"cfm": "air_flow", "ppm": "co2", "%RH": "humidity", "°F": "temperature"})
    .rename_axis(columns=None)
    .rename_axis("time")
)


## Data Cleaning & Transformation Summary

The raw CSV exported from KODE contains one row per sensor reading in long format (each row is a single timestamped value tagged with a unit). The following steps reshape it into a clean wide-format time series:

1. **Parse timestamps** — `dateTime` is converted from a string to a proper `datetime` object so it can be used as a time index.
2. **Sort and index** — rows are sorted chronologically and `dateTime` is set as the index.
3. **Drop metadata columns** — columns not needed for analysis (`pointDisplayName`, `point_id`, `dateTimeUtc`, `pointName`, `pointKind`, `deviceName`, `pointField`) are removed, leaving only `unit` and `value`.
4. **Trim to occupied window** — rows outside the class session window (07:54–10:57) are dropped to focus on the period with people in the room.
5. **Pivot to wide format** — `pivot_table` rotates each unit (`ppm`, `cfm`, `%RH`, `°F`) into its own column. Where multiple readings fall in the same second, their mean is taken.
6. **Resample to 1-minute bins** — irregular sub-minute readings are aggregated into uniform 1-minute intervals, making the time series regular.
7. **Forward-fill gaps** — sensors report at different cadences (CO2 and air flow ~every 30 s; humidity and temperature ~every few minutes). `ffill` propagates the last known reading into empty bins so every row is complete.
8. **Rename columns** — unit abbreviations are replaced with descriptive names: `air_flow` (cfm), `co2` (ppm), `humidity` (%RH), `temperature` (°F).

In [21]:
df.head(50)

,humidity,air_flow,co2,temperature
time,,,,
2026-03-29 00:00:00,21.9,1110.425659,433.5,68.360001
2026-03-29 00:01:00,21.9,1099.938599,432.0,68.360001
2026-03-29 00:02:00,21.9,1099.938599,432.0,68.360001
2026-03-29 00:03:00,21.9,1099.938599,431.0,68.360001
2026-03-29 00:04:00,21.9,1107.480591,430.0,68.360001
2026-03-29 00:05:00,21.9,1097.360352,431.5,68.360001
2026-03-29 00:06:00,21.9,1089.984741,434.0,68.360001
2026-03-29 00:07:00,21.9,1098.892578,434.0,68.360001
2026-03-29 00:08:00,21.9,1099.953979,432.0,68.360001


In [22]:
df.to_csv("data/03-28_to_04-27_rm354_FPB(cleaned).csv", index=True)

In [16]:
vb = pd.read_csv("data/4-13-2026 754am-358pm.txt", sep="\t", header=None, names=[" C8-1(v)", "C13-1(v)", "C8-7(v)", "C13-7(v)"])

In [17]:
start = pd.Timestamp("2026-04-13 07:54:00")
vb["time"] = start + pd.to_timedelta(range(len(vb)), unit="us") * 100


In [18]:
print(vb.head())

    C8-1(v)  C13-1(v)   C8-7(v)  C13-7(v)                       time
0  0.000105 -0.000426  0.002967  0.000156 2026-04-13 07:54:00.000000
1  0.000050 -0.000432  0.002936  0.000180 2026-04-13 07:54:00.000100
2  0.000051 -0.000398  0.002916  0.000163 2026-04-13 07:54:00.000200
3  0.000063 -0.000429  0.002952  0.000156 2026-04-13 07:54:00.000300
4  0.000020 -0.000442  0.002921  0.000160 2026-04-13 07:54:00.000400


In [19]:
vb.index = vb["time"]
vb = vb.drop(columns='time')

# Resample to 1-minute intervals — pick your aggregation
df_1min = vb.resample('1min').apply(lambda x: (x**2).mean()**0.5)

In [20]:
df_1min.head()

,C8-1(v),C13-1(v),C8-7(v),C13-7(v)
time,,,,
2026-04-13 07:54:00,0.001061,0.000551,0.002690,0.000534
2026-04-13 07:55:00,0.001080,0.000554,0.002699,0.000533
2026-04-13 07:56:00,0.001058,0.000537,0.002699,0.000508
2026-04-13 07:57:00,0.001046,0.000547,0.002701,0.000507
2026-04-13 07:58:00,0.000990,0.000559,0.002698,0.000512


In [21]:
start = pd.Timestamp("2026-04-13 07:54:00")
df_1min.index = df_1min.index.floor('min')

combined = df.join(df_1min, how='left')

In [22]:
print(combined.tail(50))

                      humidity     air_flow    co2  temperature   C8-1(v)  \
time                                                                        
2026-04-13 10:07:00  49.400002  1109.398071  517.0    72.319999  0.001082   
2026-04-13 10:08:00  49.400002  1117.538086  514.5    72.319999  0.001021   
2026-04-13 10:09:00  49.400002  1117.538086  515.5    72.319999  0.001133   
2026-04-13 10:10:00  49.400002  1204.961060  517.5    72.319999  0.000993   
2026-04-13 10:11:00  49.400002  1210.984070  518.0    72.319999  0.000982   
2026-04-13 10:12:00  49.400002  1210.984070  518.5    72.319999  0.000977   
2026-04-13 10:13:00  49.400002  1210.984070  512.5    72.319999  0.000978   
2026-04-13 10:14:00  49.400002  1210.984070  510.5    72.319999  0.001003   
2026-04-13 10:15:00  49.400002  1197.864136  508.5    72.319999  0.001041   
2026-04-13 10:16:00  49.400002  1214.729919  511.0    72.319999  0.001073   
2026-04-13 10:17:00  49.400002  1218.659302  510.5    72.319999  0.001073   

In [23]:
combined.to_csv("data/4-13-2026_rm354_FPB(clean)(vibration).csv", index=True)